
## Intro to PCA Notebook

This notebook uses a toy housing dataset designed so that:

The features are:

- Number of Bedrooms
- Number of Bathrooms
- Square Footage
- Number of Schools within 5 miles
- Crime Rate as a value from 1 (low crime) to 10 (high crime)


In [ ]:
#these are some important Python tools we need to load
import numpy as np #standard numpy
import matplotlib.pyplot as plt #for plotting
import pandas as pd #for looking at data
from sklearn.preprocessing import StandardScaler #for PCA
from sklearn.decomposition import PCA #for PCA

## 1. Create the dataset

In [ ]:
# -----------------------------
# STEP 1: Create DataFrame
# -----------------------------

#load data into a pandas dataframe
data = pd.DataFrame({
    "House": ["H1","H2","H3","H4","H5","H6","H7","H8","H9","H10"],
    "Bedrooms": [1,2,3,4,5,6,7,8,9,10],
    "Bathrooms": [1,2,2.5,3,3,3,4,4,4,4],
    "SquareFootage": [540.62,989.74,1503.97,2015.85,2477.78,2997.11,3501.45,3987.54,4506.00,4991.38],
    "SchoolsNearby": [5,1,9,2,8,3,7,4,6,10],
    "CrimeRate": [6,10,2,9,3,8,4,7,5,1]
})

#view the data
print(data)


## 2. Lets look at some biplots where we pick two features and try to see their relationship. 


In [ ]:
# -----------------------------
# STEP 2: Look at biplots
# -----------------------------

#bedrooms vs. bathrooms
plt.figure(figsize=(6, 4))
x = "Bedrooms"
y = "Bathrooms"
plt.scatter(data[x], data[y])
for i, row in data.iterrows():
    plt.annotate(row["House"], (row[x], row[y]),xytext=(4, 4),textcoords="offset points",fontsize=8)
plt.xlabel(x)
plt.ylabel(y)
plt.title(f"{x} vs {y}")
plt.tight_layout()
plt.show()

In [ ]:
#bathrooms vs squarefootage
plt.figure(figsize=(6, 4))
x = ???
y = ???
plt.scatter(data[x], data[y])
for i, row in data.iterrows():
    plt.annotate(row["House"], (row[x], row[y]),xytext=(4, 4),textcoords="offset points",fontsize=8)
plt.xlabel(x)
plt.ylabel(y)
plt.title(f"{x} vs {y}")
plt.tight_layout()
plt.show()

In [ ]:
#SchoolsNearby vs bedrooms
plt.figure(figsize=(6, 4))
x = ???
y = ???
plt.scatter(data[x], data[y])
for i, row in data.iterrows():
    plt.annotate(row["House"], (row[x], row[y]),xytext=(4, 4),textcoords="offset points",fontsize=8)
plt.xlabel(x)
plt.ylabel(y)
plt.title(f"{x} vs {y}")
plt.tight_layout()
plt.show()

## 2.5 Standardize the numeric features
(Secret step that's important but not THAAAT important and is really just about taking a bunch of averages to get all of the data on the same "scale".) 

In [ ]:
# -----------------------------
# STEP 2.5: Scale Data
# -----------------------------

#decide which features to standardize
features = ["Bedrooms", "Bathrooms", "SquareFootage", "SchoolsNearby", "CrimeRate"]
#set up data set with those features
X = data[features]

#choose how to scale (there are many options, stats classes will go over the pros and cons)
scaler = StandardScaler()
#scale the data
Z = scaler.fit_transform(X)

#label scaled data
labels = [f"H{i+1}" for i in range(len(data))]

#create new scaled dataframe
pd.DataFrame(Z, columns=features, index=labels).round(2)

## 3. Compute the Covariance Matrix.

In [ ]:
# -----------------------------
# STEP 3: Covariance Matrix
# -----------------------------

#feed scaled data Z into function that makes covariance matrix
#name the output cov_matrix
cov_matrix = np.cov(Z, rowvar=False)

#need this to label columns
features = ["Bedrooms", "Bathrooms", "SquareFootage", "SchoolsNearby", "CrimeRate"]

#print results
print("Covariance matrix of standardized data:")
print(pd.DataFrame(cov_matrix, index=features, columns=features))
print()

## 4. Compute the eigenvectors and eigenvalues of the COV matrix. The eigenvectors are our principal components. 

In [ ]:
# -----------------------------
# STEP 4: Eigen decomposition
# -----------------------------
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# Sort in descending order
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print("Eigenvalues (largest → smallest):")
display(pd.Series(eigenvalues, index=[f"PC{i+1}" for i in range(len(eigenvalues))]))
print()

print("Eigenvectors (columns = PCs):")
display(pd.DataFrame(
    eigenvectors,
    index=features,
    columns=[f"PC{i+1}" for i in range(len(eigenvalues))]
))

#need to save eigenvectors in a matrix for later
P = eigenvectors

## 5. Compute the explained variance 

In [ ]:
# -----------------------------
# STEP 5: Explained variance
# -----------------------------
explained = eigenvalues / eigenvalues.sum()
cumulative = np.cumsum(explained)

ev_df = pd.DataFrame({
    "Eigenvalue": eigenvalues,
    "Explained Variance Ratio": explained,
    "Cumulative Explained Variance": cumulative
}, index=[f"PC{i+1}" for i in range(len(eigenvalues))])

display(ev_df.round(4))

## 6. The data in principal component coordinates followed by a plot of the data in PC1 vs PC2 space.

In [ ]:
# -----------------------------
# STEP 6 A: Project data
# -----------------------------
scores = Z @ P

scores_df = pd.DataFrame(
    scores,
    index=[f"H{i+1}" for i in range(scores.shape[0])],
    columns=[f"PC{i+1}" for i in range(scores.shape[1])]
)

print("Data in the Principal Component coordinates:")
display(scores_df.round(3))

In [ ]:
# -----------------------------
# STEP 6 B: Plot the data in the PC1-PC2 plane
# -----------------------------
scores = Z @ P

scores_df = pd.DataFrame(
    scores,
    columns=[f"PC{i+1}" for i in range(scores.shape[1])]
)

plt.figure(figsize=(6, 4))
plt.scatter(scores_df["PC1"], scores_df["PC2"])

for i in range(len(scores_df)):
    plt.annotate(f"H{i+1}",
                 (scores_df.loc[i, "PC1"], scores_df.loc[i, "PC2"]),
                 xytext=(4, 4),
                 textcoords="offset points",
                 fontsize=8)

plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Data projected onto the PC1-PC2 plane")
plt.tight_layout()
plt.show()